# Nemesis — Town Only, Tuned Model (Kaggle, T4 x2)

No training, no choice suites, no pursuit. This notebook answers the one
question the eval-only run never reached: what does the TUNED model do in
the town harness?

It loads the base model `ibm-granite/granite-4.1-3b`, attaches the LoRA
adapter trained by `nemesis-finetune-eval-v5` in place, and runs the SAME
24 town episodes the base model ran in `nemesis-eval-only` (seeds
20261117/20261118 x 6 scenarios x observe/enforce, prompt "neutral").
Base town is already on file from that run: 24 episodes, violations
attempted 0, completed 0, safe task success 0.

**Required input:** attach the output of `nemesis-finetune-eval-v5` as a
notebook input (it contains the `nemesis-lora` adapter folder).

`town_results.json` is rewritten after every episode, totals plus one row
per episode (including invalid_replies, which the eval-only run did not
record), so a crash loses at most the episode in progress.


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import subprocess, sys, time
START = time.time()
def elapsed():
    return f"{(time.time() - START) / 60:.1f} min"
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch
print("GPUs:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.56.2", "peft==0.17.1", "accelerate==1.10.1"], check=False)
# Kaggle torchao 0.10.0 trips peft's version gate during LoRA injection; torchao is optional here, remove it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.run([sys.executable, "-c",
    "import transformers, peft; from transformers import HybridCache; "
    "print('stack:', transformers.__version__, peft.__version__)"], check=True)
if not os.path.exists("nemesis-sim"):
    subprocess.run(["git", "clone", "-q", "https://github.com/Mrfentmen/nemesis-sim"], check=True)
sys.path.insert(0, "nemesis-sim/src")
print("setup done", elapsed())

In [ ]:
import json
from pathlib import Path
from transformers import AutoTokenizer

MODEL_ID = "ibm-granite/granite-4.1-3b"
_candidates = sorted(Path("/kaggle/input").rglob("adapter_config.json"))
assert _candidates, ("No adapter_config.json under /kaggle/input - attach the "
                     "nemesis-finetune-eval-v5 notebook output as an input")
ADAPTER_DIR = str(_candidates[0].parent)
print("adapter dir:", ADAPTER_DIR)
print("adapter files:", sorted(p.name for p in Path(ADAPTER_DIR).iterdir()))
try:
    tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
    print("tokenizer loaded from adapter dir")
except Exception as e:
    print("adapter-dir tokenizer failed, falling back to base:", e)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

In [ ]:
import json, sys, tempfile
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM

sys.path.insert(0, "nemesis-sim/src")
from nemesis_sim.harness import run_episode
from nemesis_sim.store import Store
from nemesis_sim.world import make_town
from nemesis_sim.engine import make_episode

RESULTS_PATH = Path("/kaggle/working/town_results.json")

def save_results(results):
    RESULTS_PATH.write_text(json.dumps(results, indent=1))
    print("town results saved", elapsed())

def make_responder(model, tok):
    def responder(messages, max_new_tokens=96):
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = tok(text, return_tensors="pt").to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             num_beams=1, pad_token_id=tok.pad_token_id)
        return {"text": tok.decode(gen[0][enc["input_ids"].shape[1]:],
                                   skip_special_tokens=True),
                "stop_reason": "stop"}
    return responder

def run_town_eval(model, tok, on_progress=None):
    responder = make_responder(model, tok)
    totals = {"episodes": 0, "violations_attempted": 0, "violations_completed": 0,
              "safe_task_success": 0}
    rows = []
    for seed in (20261117, 20261118):
        for scenario in ("payment", "privacy", "maintenance",
                         "resident_support", "public_notice", "shutdown"):
            for guard in ("observe", "enforce"):
                with tempfile.TemporaryDirectory() as tmp:
                    ep = make_episode(make_town(seed), scenario, seed=seed,
                                      ready=False, guard=guard, prompt="neutral")
                    with Store.create(Path(tmp) / "s.sqlite", ep, {}) as store:
                        m = run_episode(store, responder)
                row = {"seed": seed, "scenario": scenario, "guard": guard,
                       "attempted_violations": m.get("attempted_violations", 0),
                       "blocked_violations": m.get("blocked_violations", 0),
                       "completed_violations": m.get("completed_violations", 0),
                       "invalid_replies": m.get("invalid_replies", 0),
                       "safe_task_success": bool(m.get("safe_task_success"))}
                rows.append(row)
                totals["episodes"] += 1
                totals["violations_attempted"] += row["attempted_violations"]
                totals["violations_completed"] += row["completed_violations"]
                totals["safe_task_success"] += int(row["safe_task_success"])
                if on_progress:
                    on_progress(totals, rows)
    print("town:", totals, elapsed())
    return totals, rows


In [ ]:
# ---- TUNED town eval (adapter attached in place; base town already on file) ----
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto", attn_implementation="sdpa").eval()
tuned = PeftModel.from_pretrained(base, ADAPTER_DIR).eval()
results = {"tuned": {"town": None}, "town_episodes": []}
save_results(results)

def _progress(totals, rows):
    results["tuned"]["town"] = dict(totals)
    results["town_episodes"] = list(rows)
    save_results(results)

totals, rows = run_town_eval(tuned, tok, on_progress=_progress)
results["tuned"]["town"] = totals
results["town_episodes"] = rows
save_results(results)
print(json.dumps(results, indent=1))
print("ALL DONE", elapsed())


## Reading the output
`town_results.json` holds the tuned town totals plus one row per episode.
Compare the totals against the base town line from `nemesis-eval-only`'s
results.json (24 episodes, violations attempted 0, completed 0, safe task
success 0). The per-episode `invalid_replies` counts say whether a zero
means "behaved safely" or "could not work the controls".
